# Simple Chain

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()

# LangChain imports
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# -----------------------------
# 1️⃣ Initialize the model
# -----------------------------
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)


# -----------------------------
# 2️⃣ Create prompt template
# -----------------------------
prompt = PromptTemplate.from_template(
"""
Explain the following programming concept in simple terms.

Concept: {concept}
"""
)


# -----------------------------
# 3️⃣ Output parser
# -----------------------------
# Converts model output to plain string
parser = StrOutputParser()


# -----------------------------
# 4️⃣ Build the chain
# -----------------------------
# LCEL pipeline
chain = prompt | model | parser


# -----------------------------
# 5️⃣ Run the chain
# -----------------------------
result = chain.invoke({
    "concept": "Hash Table"
})

print(result)

# Sequential Chain

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)

parser = StrOutputParser()

# Step 1: Summarize text
summary_prompt = PromptTemplate.from_template(
"""
Summarize the following text in 2 sentences.

{text}
"""
)

# Step 2: Convert summary to tweet
tweet_prompt = PromptTemplate.from_template(
"""
Convert this summary into a tweet.

{summary}
"""
)

# Build sequential pipeline
summary_chain = summary_prompt | model | parser
tweet_chain = tweet_prompt | model | parser

chain = summary_chain | tweet_chain

result = chain.invoke({
    "text": "LangChain is a framework that simplifies building applications using large language models."
})

print(result)

# Parallel Chain

In [ ]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# -----------------------------
# Load environment variables
# -----------------------------
load_dotenv()

# -----------------------------
# Initialize model
# -----------------------------
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
    # base_url=os.getenv("REVERSE_PROXY_HOST")
)

parser = StrOutputParser()

# -----------------------------
# Prompts
# -----------------------------
sentiment_prompt = PromptTemplate.from_template(
"Determine the sentiment of this review: {review}"
)

topic_prompt = PromptTemplate.from_template(
"Identify the main topic of this review: {review}"
)

# -----------------------------
# Parallel chain
# -----------------------------
chain = RunnableParallel(
    sentiment=sentiment_prompt | model | parser,
    topic=topic_prompt | model | parser
)

# -----------------------------
# Run chain
# -----------------------------
result = chain.invoke({
    "review": "The phone camera is amazing but the battery life is terrible."
})

print(result)

# RunnableBranch Chain

In [ ]:
# -----------------------------
# Imports
# -----------------------------
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

# -----------------------------
# Load environment variables
# -----------------------------
load_dotenv()

# -----------------------------
# Initialize model
# -----------------------------
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)

parser = StrOutputParser()


# -----------------------------
# Expert prompts
# -----------------------------
math_prompt = PromptTemplate.from_template(
"""
You are a math expert.

Solve the following problem:

{question}
"""
)

coding_prompt = PromptTemplate.from_template(
"""
You are a programming expert.

Answer the following coding question:

{question}
"""
)

general_prompt = PromptTemplate.from_template(
"""
Answer the following question:

{question}
"""
)


# -----------------------------
# Build expert chains
# -----------------------------
math_chain = math_prompt | model | parser
coding_chain = coding_prompt | model | parser
general_chain = general_prompt | model | parser


# -----------------------------
# RunnableBranch router
# -----------------------------
branch = RunnableBranch(

    # Condition → Runnable
    (lambda x: "calculate" in x["question"].lower(), math_chain),

    (lambda x: "python" in x["question"].lower(), coding_chain),

    # Default branch
    general_chain
)


# -----------------------------
# Run the chain
# -----------------------------
result = branch.invoke({
    "question": "How do I write a loop in Python?"
})

print(result)